In [ ]:
!pip install git+https://github.com/openai/whisper.git

In [ ]:
!pip install torch torchaudio noisereduce evaluate jiwer

In [ ]:
import noisereduce as nr
import soundfile as sf
import whisper
import torch
import librosa
import numpy as np
import evaluate
import re
import jiwer

In [ ]:
def clean(text):
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)
  text = re.sub(r'([а-яё])\1{2,}', r'\1', text)
  text = re.sub(r'( )\1{2,}', r'\1', text)
  text = re.sub(r'((ч|д)а(ч|д)а)+', '', text)
  text = re.sub(r'(субтитры субтитры)+', '', text)
  text = text.replace('ё', 'е')
  text = text.strip()
  text = text.replace('субтитры сделал dimatorzok', '')
  text = text.replace('субтитры создавал dimatorzok', '')
  text = text.replace('субтитры создал dimatorzok', '')
  text = text.replace('продолжение следует', '')
  text = text.replace('смотрите продолжение в следующей серии', '')
  text = text.replace('субтитры подогнал симон', '')
  text = text.replace('редактор субтитров асемкин корректор аегорова', '')
  text = text.replace('спасибо за просмотр', '')
  text = text.replace('спасибо за внимание', '')
  text = text.replace('продолжаем', '')
  text = text.replace('добро пожаловать', '')
  text = text.replace('дмитрий шепеллетов', '')
  text = text.replace('подпишись на канал и подписывайтесь на наш канал', '')
  text = text.replace('подпишись', '')
  return text

Проводим предварительное сравнение моделей на небольшой выборке аудиозаписей с объектами и с действиями (по 50 штук в каждой). Для начала подготовим эталонные транскрипции для всех аудиозаписей:

In [ ]:
audios_objects = ["Object_naming_TMS -46-1PictureProperties-4.wav", "Object_naming_TMS -46-1PictureProperties-5.wav",
          "Object_naming_TMS -46-1PictureProperties-6.wav", "Object_naming_TMS -46-1PictureProperties-7.wav",
          "Object_naming_TMS -46-1PictureProperties-8.wav", "Object_naming_TMS -46-1PictureProperties-9.wav",
          "Object_naming_TMS -46-1PictureProperties-10.wav", "Object_naming_TMS -46-1PictureProperties-11.wav",
          "Object_naming_TMS -46-1PictureProperties-12.wav", "Object_naming_TMS -46-1PictureProperties-13.wav",
          "Object_naming_TMS -46-1PictureProperties-14.wav", "Object_naming_TMS -46-1PictureProperties-15.wav",
          "Object_naming_TMS -46-1PictureProperties-16.wav", "Object_naming_TMS -46-1PictureProperties-17.wav",
          "Object_naming_TMS -46-1PictureProperties-18.wav", "Object_naming_TMS -46-1PictureProperties-19.wav",
          "Object_naming_TMS -46-1PictureProperties-20.wav", "Object_naming_TMS -46-1PictureProperties-21.wav",
          "Object_naming_TMS -46-1PictureProperties-22.wav", "Object_naming_TMS -46-1PictureProperties-23.wav",
          "Object_naming_TMS -46-1PictureProperties-24.wav", "Object_naming_TMS -46-1PictureProperties-25.wav",
          "Object_naming_TMS -46-1PictureProperties-26.wav", "Object_naming_TMS -46-1PictureProperties-27.wav",
          "Object_naming_TMS -46-1PictureProperties-28.wav", "Object_naming_TMS -46-1PictureProperties-29.wav",
          "Object_naming_TMS -46-1PictureProperties-30.wav", "Object_naming_TMS -46-1PictureProperties-31.wav",
          "Object_naming_TMS -46-1PictureProperties-32.wav", "Object_naming_TMS -46-1PictureProperties-33.wav",
          "Object_naming_TMS -46-1PictureProperties-34.wav", "Object_naming_TMS -46-1PictureProperties-35.wav",
          "Object_naming_TMS -46-1PictureProperties-36.wav", "Object_naming_TMS -46-1PictureProperties-37.wav",
          "Object_naming_TMS -46-1PictureProperties-38.wav", "Object_naming_TMS -46-1PictureProperties-39.wav",
          "Object_naming_TMS -46-1PictureProperties-40.wav", "Object_naming_TMS -46-1PictureProperties-41.wav",
          "Object_naming_TMS -46-1PictureProperties-42.wav", "Object_naming_TMS -46-1PictureProperties-43.wav",
          "Object_naming_TMS -46-1PictureProperties-44.wav", "Object_naming_TMS -46-1PictureProperties-45.wav",
          "Object_naming_TMS -46-1PictureProperties-46.wav", "Object_naming_TMS -46-1PictureProperties-47.wav",
          "Object_naming_TMS -46-1PictureProperties-48.wav", "Object_naming_TMS -46-1PictureProperties-49.wav",
          "Object_naming_TMS -46-1PictureProperties-50.wav", "Object_naming_TMS -46-1PictureProperties-51.wav",
          "Object_naming_TMS -46-1PictureProperties-52.wav", "Object_naming_TMS -46-1PictureProperties-53.wav"]

In [ ]:
references_objects = ["Это банка", "Это кроссворд", "Это клоун", "Это верблюд", "Это кольцо", "Это пружина", "Это домино", "Это перчатка", "Это аквариум", "Это наручники",
                      "Это пистолет", "Это кастрю- кастрюля", "Это страус", "Это корзинка", "Это тюльпан", "Это кактус", "Это вешалка", "Это цепь", "Это коляска",
                      "Это скрепка", "Это помидор", "Это жираф", "Это барабан", "Это башня", "Это вертолёт", "Это щётка", "Это фен", "Это микрофон", "Это ракета",
                      "Это лягушка", "Это пылесос", "Это ключ", "Это шляпа", "Это костёр", "Это маска", "Это свитер", "Это кенгуру", "Это водопад", "Это зажигалка",
                      "Это горох", "Это бегемот", "Это кресло", "Это костюм", "Это бутерброд", "Это брюки", "Это мешок", "Это пиджак", "Это кровать", "Это краска",
                      "Это качели"]

In [ ]:
references_objects = list(map(clean, references_objects))
print(references_objects)

['это банка', 'это кроссворд', 'это клоун', 'это верблюд', 'это кольцо', 'это пружина', 'это домино', 'это перчатка', 'это аквариум', 'это наручники', 'это пистолет', 'это кастрю кастрюля', 'это страус', 'это корзинка', 'это тюльпан', 'это кактус', 'это вешалка', 'это цепь', 'это коляска', 'это скрепка', 'это помидор', 'это жираф', 'это барабан', 'это башня', 'это вертолет', 'это щетка', 'это фен', 'это микрофон', 'это ракета', 'это лягушка', 'это пылесос', 'это ключ', 'это шляпа', 'это костер', 'это маска', 'это свитер', 'это кенгуру', 'это водопад', 'это зажигалка', 'это горох', 'это бегемот', 'это кресло', 'это костюм', 'это бутерброд', 'это брюки', 'это мешок', 'это пиджак', 'это кровать', 'это краска', 'это качели']


In [ ]:
audios_actions = ["/content/Action_naming_TMS-15-2-1PictureProperties-1.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-2.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-3.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-4.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-5.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-6.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-7.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-8.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-9.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-10.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-11.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-12.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-13.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-14.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-15.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-16.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-17.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-18.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-19.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-20.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-21.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-22.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-23.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-24.wav",
                  "/content/Action_naming_TMS-15-2-1PictureProperties-25.wav", "/content/Action_naming_TMS-15-2-1PictureProperties-26.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-1.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-2.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-3.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-4.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-5.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-6.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-7.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-8.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-9.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-10.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-11.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-12.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-13.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-14.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-15.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-16.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-17.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-18.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-19.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-20.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-21.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-22.wav",
                  "/content/Action_naming_TMS-15-2-2PictureProperties-23.wav", "/content/Action_naming_TMS-15-2-2PictureProperties-24.wav"]

In [ ]:
references_actions = ["Тут мальчик плывёт", "Тут дочка пишет", "Тут птица летит", "Тут пожарный тушит", "Тут рабочий сверлит", "Тут бабушка думает",
                      "Тут сестра качается", "Тут юноша бреется", "Тут мальчик надувает", "Тут девочка нюхает", "Тут брат плачет. Сейчас тоже плохо? ", "Тут малыш играет. И сейчас, да?",
                      "Тут мама пылесосит", "Тут тётя режет", "Тут мама гладит. Сейчас тоже", "Тут папа стрижёт", "Тут корова мычит", "Тут", "Тут артист поёт",
                      "Тут бабушка кормит", "Тут пара танцует", "Тут дядя копает", "Тут девочка кушает", "Тут малыш падает", "Тут бабушка трёт", "Тут гость стучит",
                      "Тут мальчик плывёт", "Тут дочка пишет", "Тут птица летит. Подвожу катушку", "Тут сын наряжает", "Тут старушка доит", "Тут папа зажигает",
                      "Тут мальчик слушает", "Тут самолёт взле- взлетает", "Тут рабочий косит", "Тут старик сеет", "Тут птица вылетает", "Тут брат ловит",
                      "Тут рабочий пилит", "Тут дочка режет", "Тут сестра болеет", "Тут юноша красит", "Тут барабанщик стучит", "Тут до- до- дочка чистит",
                      "Тут супруги целуются", "Тут матрос плывёт", "Тут няня моет", "Тут старушка поливает", "Тут юноша ждёт", "Тут брат собирает"]

In [ ]:
references_actions = list(map(clean, references_actions))
print(references_actions)

['тут мальчик плывет', 'тут дочка пишет', 'тут птица летит', 'тут пожарный тушит', 'тут рабочий сверлит', 'тут бабушка думает', 'тут сестра качается', 'тут юноша бреется', 'тут мальчик надувает', 'тут девочка нюхает', 'тут брат плачет сейчас тоже плохо', 'тут малыш играет и сейчас да', 'тут мама пылесосит', 'тут тетя режет', 'тут мама гладит сейчас тоже', 'тут папа стрижет', 'тут корова мычит', 'тут', 'тут артист поет', 'тут бабушка кормит', 'тут пара танцует', 'тут дядя копает', 'тут девочка кушает', 'тут малыш падает', 'тут бабушка трет', 'тут гость стучит', 'тут мальчик плывет', 'тут дочка пишет', 'тут птица летит подвожу катушку', 'тут сын наряжает', 'тут старушка доит', 'тут папа зажигает', 'тут мальчик слушает', 'тут самолет взле взлетает', 'тут рабочий косит', 'тут старик сеет', 'тут птица вылетает', 'тут брат ловит', 'тут рабочий пилит', 'тут дочка режет', 'тут сестра болеет', 'тут юноша красит', 'тут барабанщик стучит', 'тут до до дочка чистит', 'тут супруги целуются', 'тут ма

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model_base = whisper.load_model("base").to(device)

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 113MiB/s]


In [ ]:
base_obj_predictions = []

for audio in audios_objects:
  result = model_base.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  base_obj_predictions.append(result["text"])

  #выводит время в мс, через которое на зписи прозвучал токен под указанным индексом
  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

680.0  Это больно.
680.0  Это кроссово.

600.0  Вот и вот и вот и вот.
980.0  Тихо, тихо, тихо.
0.0  Продолжаем. Продолжаем.
1020.0  Попробуй моему.
540.0  Это подчерк.
0.0  Потоковарим.
700.0  Это наружие.
680.0  Попробуйте стрелять.
780.0  Это пустя пустя пустя...
740.0  Какой-то взбороз.
740.0  Это коробие.
780.0  Ох, ты бомб!
700.0  Это как-то.
820.0  Добро пожаловать!

800.0  Попробуй вас.
700.0  Ну, ступи.
0.0  Попробуйте. Попробуйте.
740.0  Попробуй его.
780.0  Это бурабан.
720.0  Это плачка.
620.0  Это не то, что делать.
680.0  Это что?
700.0  Это что?
620.0  Это не каракон.
0.0  Отверстие.
620.0  Вот теперь пришка.
740.0  Это поле соус.
640.0  Вот так вот.
600.0  Это же самое.
660.0  Вот это что
820.0  Это мой.
540.0  А это все.
680.0  Это пункт урок.
760.0  вот это вот и под
740.0  Это лепебелка.
0.0  Попробуй. Попробуй.
700.0  Вот это уже вот
0.0  Отдохнился.
720.0  Это кастрюлю.
1000.0  Наконки рубро.
840.0  Как это было? Как это было?
860.0  Вот и на шоу.

560.0  Ну-ка, по

In [ ]:
base_act_predictions = []

for audio in audios_actions:
  result = model_base.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  base_act_predictions.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

960.0  Дмитрий Шепеллётов.


1000.0  Проверим тушить.
720.0  и там очень вслепно
940.0  и бабушка думает
1040.0  Иисыка,аааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааааа Аааааааааааааааааааааааааааааа
940.0  Тот же уедем на шагеринцию.
820.0  Это очень удобно.
960.0  тут девочке не уходят
660.0  Тот и тракт плачет.
1160.0  Молочь играет.
760.0  Тут мало пересосит.
780.0  Тупер серебрт.
1040.0  Тут Моу и кладят. Сейчас тоже.
700.0  и по поводу стрелки.
1340.0  Кроме чуть-чуть.
860.0  Думаю, ты...
960.0  Куперти спает
1000.0  Тут бабочка и коридит
860.0  и пора консульт
1000.0  и дядя копает
900.0  и девочка кушает
880.0  Тихо, может быть, падает.
800.0  Туда мы скаем и трет.
680.0  Кто-то поздно чит?
940.0  Мочек нет.
820.0  тут дочкой пишут

760.0  Такси нарежет.
660.0  Друзья, здоровья.
920.0  Тут папа и дожигать.
1020

In [ ]:
base_obj_predictions = list(map(clean, base_obj_predictions))
print(base_obj_predictions)

['это больно', 'это кроссово', '', 'вот и вот и вот и вот', 'тихо тихо тихо', ' ', 'попробуй моему', 'это подчерк', 'потоковарим', 'это наружие', 'попробуйте стрелять', 'это пустя пустя пустя', 'какойто взбороз', 'это коробие', 'ох ты бомб', 'это както', '', '', 'попробуй вас', 'ну ступи', 'попробуйте попробуйте', 'попробуй его', 'это бурабан', 'это плачка', 'это не то что делать', 'это что', 'это что', 'это не каракон', 'отверстие', 'вот теперь пришка', 'это поле соус', 'вот так вот', 'это же самое', 'вот это что', 'это мой', 'а это все', 'это пункт урок', 'вот это вот и под', 'это лепебелка', 'попробуй попробуй', 'вот это уже вот', 'отдохнился', 'это кастрюлю', 'наконки рубро', 'как это было как это было', 'вот и на шоу', '', 'нука попробуйте', 'это просто', 'это качание']


In [ ]:
base_act_predictions = list(map(clean, base_act_predictions))
print(base_act_predictions)

['', '', '', 'проверим тушить', 'и там очень вслепно', 'и бабушка думает', 'иисыка а', 'тот же уедем на шагеринцию', 'это очень удобно', 'тут девочке не уходят', 'тот и тракт плачет', 'молочь играет', 'тут мало пересосит', 'тупер серебрт', 'тут моу и кладят сейчас тоже', 'и по поводу стрелки', 'кроме чутьчуть', 'думаю ты', 'куперти спает', 'тут бабочка и коридит', 'и пора консульт', 'и дядя копает', 'и девочка кушает', 'тихо может быть падает', 'туда мы скаем и трет', 'ктото поздно чит', 'мочек нет', 'тут дочкой пишут', '', 'такси нарежет', 'друзья здоровья', 'тут папа и дожигать', 'и больше больше', 'тот что он надел тебя взлетает', 'тут рабочий хвост', 'у старих сейт', 'оттикся отмолитаем', 'доброе утро', 'тут рабочий билет', 'тут дочь принято', 'это стераболит', 'тут я не чаю красить', 'вон чтото случилось', 'тут дочь очистить', 'я не знаю я не знаю я не знаю я не знаю', 'идем и просто елто', 'и в ней не будет', 'и вторым плевает', 'чтото емно что ждет', 'брат собирает']


In [ ]:
obj_wer = wer_metric.compute(predictions=base_obj_predictions, references=references_objects)
act_wer = wer_metric.compute(predictions=base_act_predictions, references=references_actions)
obj_cer = cer_metric.compute(predictions=base_obj_predictions, references=references_objects)
act_cer = cer_metric.compute(predictions=base_act_predictions, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 104.0%
act_wer: 100.0%
obj_cer: 77.4%
act_cer: 62.6%


In [ ]:
model_small = whisper.load_model("small").to(device)

100%|███████████████████████████████████████| 461M/461M [00:07<00:00, 62.8MiB/s]


In [ ]:
small_obj_predictions = []

for audio in audios_objects:
  result = model_small.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  small_obj_predictions.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

660.0  Это больно
680.0  Это кроссовый.

620.0  Это верблю.
600.0  Что-то есть.
1080.0  Продолжение следует
840.0  это вы мой мой
560.0  Это перчатка
720.0  это аквариум
700.0  Это наручники
540.0  это пистолет
760.0  это пустырь пустырь
740.0  Это скроз
720.0  Это корзина
440.00000000000006  Почти. Почти.
720.0  Это как-то...
640.0  Что ты делаешь?

600.0  Это ты лёгкий
680.0  Let's do it.
880.0  Подпишись на канал и подписывайтесь на наш канал!
700.0  Удачи, удачи
780.0  Это пробан
720.0  это пошло
640.0  это вертоза
680.0  Это что?
700.0  Это всё.
640.0  Это микрофон
960.0  Поперок нет
600.0  У тебя пушка
720.0  Это пылесос.
660.0  Вот так беш.
600.0  Это что?
660.0  Ну, теперь всё.
780.0  Что это, Макс?
740.0  Та-а-а-а-а!
680.0  Это не бро
800.0  Это водопад
720.0  А то зажигалка
720.0  Это горло.
760.0  Это ведь его вот
780.0  Это колес
720.0  Это то стрюбил.
700.0  Я не могу тебя брать
840.0  Ну что ты делаешь?
940.0  Очень хорошо.
0.0  Фикшер
600.0  Надо попробовать.
700.0  Это 

In [ ]:
small_act_predictions = []

for audio in audios_actions:
  result = model_small.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  small_act_predictions.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1060.0  Как мальчик уехал
900.0  Подпишись, подпишись.
0.0  Спасибо.
1040.0  Пожарный тушит
1140.0  Тут рабочие вследнадцать.
960.0  Тут бабушка думает
1320.0  Пусть устраивает шансы.
880.0  То есть, у него что греется.
880.0  Тут мальчик называют
920.0  Тут девочка нюхает
660.0  тут и рад плачет сейчас тоже трудно
1160.0  Вот малыш играет
760.0  Тут мама пересосит
800.0  тут же серия
960.0  тут мало кладет сейчас тоже
1000.0  Тут папа, где стрежок.
1000.0  Пусть корову начитит.
880.0  Та-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да-да- Субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субтитры субт

In [ ]:
small_obj_predictions = list(map(clean, small_obj_predictions))
print(small_obj_predictions)

['это больно', 'это кроссовый', '', 'это верблю', 'чтото есть', '', 'это вы мой мой', 'это перчатка', 'это аквариум', 'это наручники', 'это пистолет', 'это пустырь пустырь', 'это скроз', 'это корзина', 'почти почти', 'это както', 'что ты делаешь', '', 'это ты легкий', 'lets do it', '', 'удачи удачи', 'это пробан', 'это пошло', 'это вертоза', 'это что', 'это все', 'это микрофон', 'поперок нет', 'у тебя пушка', 'это пылесос', 'вот так беш', 'это что', 'ну теперь все', 'что это макс', 'та', 'это не бро', 'это водопад', 'а то зажигалка', 'это горло', 'это ведь его вот', 'это колес', 'это то стрюбил', 'я не могу тебя брать', 'ну что ты делаешь', 'очень хорошо', 'фикшер', 'надо попробовать', 'это краска', 'это кочей']


In [ ]:
small_act_predictions = list(map(clean, small_act_predictions))
print(small_act_predictions)

['как мальчик уехал', '', 'спасибо', 'пожарный тушит', 'тут рабочие вследнадцать', 'тут бабушка думает', 'пусть устраивает шансы', 'то есть у него что греется', 'тут мальчик называют', 'тут девочка нюхает', 'тут и рад плачет сейчас тоже трудно', 'вот малыш играет', 'тут мама пересосит', 'тут же серия', 'тут мало кладет сейчас тоже', 'тут папа где стрежок', 'пусть корову начитит', 'та с', 'тут артист поет', 'тут бабушка кормит', 'тут пара консультов', 'чай чай копает', 'и девочку кушать', 'тут может падать', 'тут дабы скажем трято', 'тут кость кричит', 'мальчик привет', 'ты дочка пишет', 'попчисклетит плажем карточку', 'чтото стыдно рожает', 'пусть наружка доведет', 'тут папа и держи гайку', 'тут мальчик спушит', 'тут он на тебя взлетает', 'тут рабочий кост', 'все все все все все все все все', 'тут птица отлетает', 'ктото проработает', 'тут рабочий пилит', 'тут дочка режет', 'чтото страшно болеет', 'тут все ниче не окрасит', 'путбай мальчик стучите', 'тут дочка чистит', 'пусть украдется

In [ ]:
obj_wer = wer_metric.compute(predictions=small_obj_predictions, references=references_objects)
act_wer = wer_metric.compute(predictions=small_act_predictions, references=references_actions)
obj_cer = cer_metric.compute(predictions=small_obj_predictions, references=references_objects)
act_cer = cer_metric.compute(predictions=small_act_predictions, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 80.2%
act_wer: 68.3%
obj_cer: 54.2%
act_cer: 39.8%


In [ ]:
model_medium = whisper.load_model("medium").to(device)

100%|█████████████████████████████████████| 1.42G/1.42G [00:27<00:00, 56.5MiB/s]


In [ ]:
medium_obj_predictions = []

for audio in audios_objects:
  result = model_medium.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  medium_obj_predictions.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

720.0  Вот так вот.
820.0  Это красава!
980.0  Спасибо за внимание!
720.0  Это игр, блядь.
760.0  Вот это и все.
740.0  Это пружина.
940.0  Это дымом
640.0  Это перчатка.
840.0  Это аквариум.
860.0  Это наручники.
720.0  Пути пистолет
940.0  Это Костюм. Костюм.

1060.0  Доктор Зин


720.0  Чё ты бьёшь?

660.0  Вот так-то лёгко.



900.0  Это барабан
880.0  Это башня.
700.0  Это вертолет.
860.0  Это что?
840.0  Это всё.
780.0  Это микрофон.
880.0  Мотор отъедет.
700.0  Вот тебе кушка.
820.0  Это пылесос.
720.0  Вот и ключ.
700.0  это же смертно




880.0  Это водопад.

820.0  Это здоровье.
820.0  Вот так вот
900.0  Это перес.
780.0  Вот это стрельба


960.0  вот ему шок
1080.0  Фик-фик-фик-фик-фик-фик
0.0  Попробую
820.0  Это краска.
840.0  Вот и качали.


In [ ]:
medium_act_predictions = []

for audio in audios_actions:
  result = model_medium.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  medium_act_predictions.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1140.0  Тут мальчик ходит
1020.0  В точке пишет.
2380.0  Спасибо за внимание!
1120.0  Пожар не тушит
1260.0  И рабочие следуют.
1020.0  Бабушка думает
1420.0  Пусть устраивает счастье.
1480.0  Я думаю, что греется.
980.0  Тут мальчик называет.
1000.0  Тут девочка нюхает
800.0  Кто-то рад, плачет. Сейчас тоже трудно?
1280.0  Малыш играет
900.0  Тут мало колесосит.

1100.0  тут мало класит
1080.0  Тут пока стрижок.
1040.0  Тут корова мучит.

1140.0  Тут артист поет.
1160.0  Тут бабушка кормит.
1040.0  Тут пара консульт
1020.0  Тут чай звера копает
1040.0  Девочка кушает
900.0  Тут новый шпад
1080.0  Тут бабушка в этот ряд
980.0  Тут кость звучит.
1000.0  мальчик хуят
900.0  Тут точка пишет

920.0  Тут сыр не наряжает.
1080.0  Ту старушку дует
1120.0  Тут папа зажигает
960.0  Тут мальчик слушает
1500.0  Тут самолёт взлетает.
1100.0  Тут рабочий кост.
1060.0  Тут старик всеет
1100.0  тут птица летает
1000.0  Тут забраться будет
1060.0  Тут рабочий пилит.
960.0  Тут дочка режет
1160.0  Сест

In [ ]:
medium_obj_predictions = list(map(clean, medium_obj_predictions))
print(medium_obj_predictions)

['вот так вот', 'это красава', '', 'это игр блядь', 'вот это и все', 'это пружина', 'это дымом', 'это перчатка', 'это аквариум', 'это наручники', 'пути пистолет', 'это костюм костюм', '', 'доктор зин', '', '', 'че ты бьешь', '', 'вот такто легко', '', '', '', 'это барабан', 'это башня', 'это вертолет', 'это что', 'это все', 'это микрофон', 'мотор отъедет', 'вот тебе кушка', 'это пылесос', 'вот и ключ', 'это же смертно', '', '', '', '', 'это водопад', '', 'это здоровье', 'вот так вот', 'это перес', 'вот это стрельба', '', '', 'вот ему шок', 'фикфикфикфикфикфик', 'попробую', 'это краска', 'вот и качали']


In [ ]:
medium_act_predictions = list(map(clean, medium_act_predictions))
print(medium_act_predictions)

['тут мальчик ходит', 'в точке пишет', '', 'пожар не тушит', 'и рабочие следуют', 'бабушка думает', 'пусть устраивает счастье', 'я думаю что греется', 'тут мальчик называет', 'тут девочка нюхает', 'ктото рад плачет сейчас тоже трудно', 'малыш играет', 'тут мало колесосит', '', 'тут мало класит', 'тут пока стрижок', 'тут корова мучит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', 'тут чай звера копает', 'девочка кушает', 'тут новый шпад', 'тут бабушка в этот ряд', 'тут кость звучит', 'мальчик хуят', 'тут точка пишет', '', 'тут сыр не наряжает', 'ту старушку дует', 'тут папа зажигает', 'тут мальчик слушает', 'тут самолет взлетает', 'тут рабочий кост', 'тут старик всеет', 'тут птица летает', 'тут забраться будет', 'тут рабочий пилит', 'тут дочка режет', 'сестра болеет', 'тут все лучше окрасить', 'тут бы мальчик стучит', 'дочка чистит', 'пусть у тебя будет солнце', 'тут матрас царят', 'тут меня не моет', '', 'тут юношка ждет', 'тут брат собирает']


In [ ]:
obj_wer = wer_metric.compute(predictions=medium_obj_predictions, references=references_objects)
act_wer = wer_metric.compute(predictions=medium_act_predictions, references=references_actions)
obj_cer = cer_metric.compute(predictions=medium_obj_predictions, references=references_objects)
act_cer = cer_metric.compute(predictions=medium_act_predictions, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 78.2%
act_wer: 57.1%
obj_cer: 61.7%
act_cer: 36.2%


In [ ]:
model_large = whisper.load_model("large").to(device)

100%|█████████████████████████████████████| 2.88G/2.88G [00:39<00:00, 79.1MiB/s]


In [ ]:
large_obj_predictions = []

for audio in audios_objects:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_obj_predictions.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

620.0  Это было.
620.0  это кроссворд
1060.0  Субтитры сделал DimaTorzok
540.0  Это верблюд.
520.0  Вот так вот.
500.0  Это пружина.
780.0  Это вымыл он.
480.0  это перчатка
660.0  Это аквариум.
620.0  Это наручники.
420.0  Тут и пистолет.
700.0  Это кастыль.
680.0  Это страус.
620.0  Это корзина.
840.0  Субтитры сделал DimaTorzok
660.0  Это кактус.
640.0  как ты видишь
3500.0  Субтитры сделал DimaTorzok
540.0  это колеса
3440.0  Субтитры сделал DimaTorzok
940.0  Продолжение следует...
520.0  Это же рэп.
740.0  Это барабан.
660.0  Это башня.
540.0  Это вертолёт.
600.0  Это шоу.
640.0  Это фильм.
560.0  Это микрофон.
920.0  Поперек нет.
600.0  Вот тебе кружка.
660.0  Это пылесос.
600.0  Это ключ.
540.0  Это все.
660.0  Вот такая ситуация.
760.0  Это МАС.
660.0  Это суть.
620.0  Это кунгур.
720.0  Это водопад.
820.0  оттуда уже белка
640.0  Это дорога.
900.0  Субтитры сделал DimaTorzok
740.0  Это крёст.
640.0  Это кастрюля.
960.0  Продолжение следует...
1300.0  Субтитры сделал DimaTorzok

In [ ]:
large_act_predictions = []

for audio in audios_actions:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_act_predictions.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1040.0  Мальчик улет.
900.0  И точка пишет.
1520.0  Субтитры сделал DimaTorzok
980.0  Пожарный тушит.
1060.0  тут рабочие вслепят
860.0  Тут бабушка думает.
540.0  тут сестра
880.0  Так, я думаю, что греется.
820.0  тут мальчик надувает
860.0  Тут девочка нюхает.
980.0  Тут тетрадь плачет. Сейчас тоже тетрадь?
1060.0  Тут малыш играет. И сейчас, да?
680.0  тут мама пылесосит
760.0  Тут все серебро.
840.0  Тут мало кладет. Сейчас тоже.
820.0  Тут папа стрижет.
880.0  Тут корова мочит.
1240.0  Субтитры сделал DimaTorzok
1040.0  Тут артист поет.
920.0  Тут бабушка кормит.
820.0  Тут пара консульт.
1000.0  Ча-ча-ча-ча-ча-ча
520.0  Какая-то девочка кушает.
840.0  тут малыш падает
900.0  Тут бабушка в этот ряд.
880.0  Тут кость кричит.
840.0  Мальчик пьет.
680.0  Тут дочка пищит.
2640.0  Смотрите продолжение в следующей серии.
540.0  Что-то со мной наряжается.
840.0  Пусторушка дует.
860.0  Тут папа зажигает.
840.0  Тут мальчик слушает.
1020.0  Тут сломанная лёздовица летает.
920.0  Тут рабо

In [ ]:
large_obj_predictions = list(map(clean, large_obj_predictions))
print(large_obj_predictions)

['это было', 'это кроссворд', '', 'это верблюд', 'вот так вот', 'это пружина', 'это вымыл он', 'это перчатка', 'это аквариум', 'это наручники', 'тут и пистолет', 'это кастыль', 'это страус', 'это корзина', '', 'это кактус', 'как ты видишь', '', 'это колеса', '', '', 'это же рэп', 'это барабан', 'это башня', 'это вертолет', 'это шоу', 'это фильм', 'это микрофон', 'поперек нет', 'вот тебе кружка', 'это пылесос', 'это ключ', 'это все', 'вот такая ситуация', 'это мас', 'это суть', 'это кунгур', 'это водопад', 'оттуда уже белка', 'это дорога', '', 'это крест', 'это кастрюля', '', '', 'вот и мы в шоке', 'это пикшок', 'попробовать', 'это краска', 'это качай']


In [ ]:
large_act_predictions = list(map(clean, large_act_predictions))
print(large_act_predictions)

['мальчик улет', 'и точка пишет', '', 'пожарный тушит', 'тут рабочие вслепят', 'тут бабушка думает', 'тут сестра', 'так я думаю что греется', 'тут мальчик надувает', 'тут девочка нюхает', 'тут тетрадь плачет сейчас тоже тетрадь', 'тут малыш играет и сейчас да', 'тут мама пылесосит', 'тут все серебро', 'тут мало кладет сейчас тоже', 'тут папа стрижет', 'тут корова мочит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', '', 'какаято девочка кушает', 'тут малыш падает', 'тут бабушка в этот ряд', 'тут кость кричит', 'мальчик пьет', 'тут дочка пищит', '', 'чтото со мной наряжается', 'пусторушка дует', 'тут папа зажигает', 'тут мальчик слушает', 'тут сломанная лездовица летает', 'тут рабочий кост', 'тут старик сеет', 'тут птица летает', 'тут и брат плавит', 'питер больше пилит', 'тут дочка режет', 'тут сестра болеет', 'тут фигуру еще накрасить', 'судьба мальчику стучит', 'тут дочка чистит', '', 'тут матрас стоит', 'тут меня не моют', 'и старушка плевает', 'тут кивнушка ждет

In [ ]:
obj_wer = wer_metric.compute(predictions=large_obj_predictions, references=references_objects)
act_wer = wer_metric.compute(predictions=large_act_predictions, references=references_actions)
obj_cer = cer_metric.compute(predictions=large_obj_predictions, references=references_objects)
act_cer = cer_metric.compute(predictions=large_act_predictions, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 61.4%
act_wer: 47.2%
obj_cer: 43.2%
act_cer: 31.4%


Видим, что самые низкие показатели ошибок получаются при помощи large модели. Попробуем взять эту модель и поизменять гиперпараметры при её запуске - возможно, удастся ещё улучшить результаты

Сначала температура:

In [ ]:
large_obj_predictions1 = []

for audio in audios_objects:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.2,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_obj_predictions1.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

620.0  Это было.
620.0  это кроссворд
660.0  Что такое?
540.0  Это верблюд.
520.0  Вот так вот.
1040.0  Продолжение следует...
780.0  Это дым и лом.
480.0  это перчатка
660.0  Это аквариум.
620.0  Это наручники.
480.0  Это пистолет.
700.0  Это кастыль.
680.0  Это страус.
620.0  Это корзина.
620.0  Это фитбол.
660.0  Это кактус.
640.0  как ты видишь
0.0  Спасибо.
540.0  это колеса
3440.0  Субтитры сделал DimaTorzok
660.0  Это примерно 2.
520.0  Это же рэп.
740.0  Это барабан.
660.0  Это башня.
540.0  Это вертолёт.
600.0  Это шоу.
640.0  Это фильм.
560.0  Это микрофон.
920.0  Поперек нет.
600.0  Вот тебе кружка.
660.0  Это пылесос.
600.0  Это ключ.
540.0  Это все.
620.0  Это как-то так.
760.0  Это МАС.
660.0  Это суть.
620.0  Это кунгур.
720.0  Это водопад.
820.0  оттуда уже белка
640.0  Это дорога.
880.0  Продолжение следует...
740.0  Это крёст.
640.0  Это то стрельба.
540.0  Я помню, ты его брал.
1300.0  Субтитры сделал DimaTorzok
860.0  Вот и мы в шоке.
780.0  Это пикшот.
0.0  попробо

In [ ]:
large_act_predictions1 = []

for audio in audios_actions:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.2,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      condition_on_previous_text=False,
      word_timestamps=True
  )

  large_act_predictions1.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1040.0  Мальчик улет.
760.0  Точка пишет.
1520.0  Субтитры сделал DimaTorzok
980.0  Пожарный тушит.
0.0  рабочие
860.0  Тут бабушка думает.
540.0  тут сестра
1460.0  только вернусь в границу
820.0  тут мальчик надувает
860.0  Тут девочка нюхает.
980.0  Тут тетрадь плачет. Сейчас тоже тетрадь?
1060.0  Тут малыш играет. И сейчас, да?
680.0  тут мама пылесосит
760.0  Тут все серебро.
840.0  Тут мало кладет. Сейчас тоже.
780.0  тут папа стрижет
880.0  Тут корова мочит.
1240.0  Субтитры сделал DimaTorzok
1040.0  Тут артист поет.
920.0  Тут бабушка кормит.
820.0  Тут пара консульт.
1040.0  Ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ч Продолжение следует...
840.0  девочка кушает
840.0  тут малыш падает
900.0  Тут бабушка в этот ряд.
880.0  Тут кость кричит.
840.0  Мальчик пьет.
680.0  Тут дочка пищит.
0.0  Светит.
54

In [ ]:
large_obj_predictions1 = list(map(clean, large_obj_predictions1))
print(large_obj_predictions1)

['это было', 'это кроссворд', 'что такое', 'это верблюд', 'вот так вот', '', 'это дым и лом', 'это перчатка', 'это аквариум', 'это наручники', 'это пистолет', 'это кастыль', 'это страус', 'это корзина', 'это фитбол', 'это кактус', 'как ты видишь', 'спасибо', 'это колеса', '', 'это примерно 2', 'это же рэп', 'это барабан', 'это башня', 'это вертолет', 'это шоу', 'это фильм', 'это микрофон', 'поперек нет', 'вот тебе кружка', 'это пылесос', 'это ключ', 'это все', 'это както так', 'это мас', 'это суть', 'это кунгур', 'это водопад', 'оттуда уже белка', 'это дорога', '', 'это крест', 'это то стрельба', 'я помню ты его брал', '', 'вот и мы в шоке', 'это пикшот', 'попробовать', 'это краска', 'это качай']


In [ ]:
large_act_predictions1 = list(map(clean, large_act_predictions1))
print(large_act_predictions1)

['мальчик улет', 'точка пишет', '', 'пожарный тушит', 'рабочие', 'тут бабушка думает', 'тут сестра', 'только вернусь в границу', 'тут мальчик надувает', 'тут девочка нюхает', 'тут тетрадь плачет сейчас тоже тетрадь', 'тут малыш играет и сейчас да', 'тут мама пылесосит', 'тут все серебро', 'тут мало кладет сейчас тоже', 'тут папа стрижет', 'тут корова мочит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', 'ч ', 'девочка кушает', 'тут малыш падает', 'тут бабушка в этот ряд', 'тут кость кричит', 'мальчик пьет', 'тут дочка пищит', 'светит', 'чтото с ними наряжается', 'пусторус подует', 'тут папа зажигает', 'тут мальчик слушает', 'тут сломанная лездовица летает', 'тут рабочий кост', 'тут старик сеет', 'тут птица летает', 'тут и брат плавит', 'питер больше пилит', 'тут дочка режет', 'тут все старо болеет', 'тут фигуру еще накрасить', 'тут бы обманщик стучит', 'тут дочка чистит', 'пусть собрали целую цепь', 'тут матрас стоит', 'тут меня не моют', 'и старушка плевает', 'тут 

In [ ]:
obj_wer = wer_metric.compute(predictions=large_obj_predictions1, references=references_objects)
act_wer = wer_metric.compute(predictions=large_act_predictions1, references=references_actions)
obj_cer = cer_metric.compute(predictions=large_obj_predictions1, references=references_objects)
act_cer = cer_metric.compute(predictions=large_act_predictions1, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 64.4%
act_wer: 49.1%
obj_cer: 41.9%
act_cer: 31.3%


In [ ]:
large_obj_predictions2 = []

for audio in audios_objects:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.4,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_obj_predictions2.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

620.0  Это бан.
620.0  это кроссворд
880.0  Продолжение следует...
540.0  Это верблюд.
520.0  Вот так вот.
1040.0  Продолжение следует...
780.0  Это дым и лом.
480.0  это перчатка
660.0  Это аквариум.
620.0  Это наручники.
480.0  Это пистолет.
700.0  Это кастыль.
680.0  Это страус.
620.0  Это корзина.

660.0  Это кактус.
640.0  как ты видишь
3500.0  Субтитры сделал DimaTorzok
540.0  это колеса
3440.0  Субтитры сделал DimaTorzok
660.0  Это примерно 2.
520.0  Это же рэп.
740.0  Это барабан.
660.0  Это башня.
540.0  Это вертолет.
1020.0  Субтитры сделал DimaTorzok
640.0  Это фильм.
560.0  Это микрофон.
920.0  Поперек нет.
600.0  Вот тебе кружка.
660.0  Это пылесос.
600.0  Это ключ.
540.0  Это все.
620.0  Это как-то так.
760.0  Это МАС.
660.0  Это суть.
620.0  Это кунгур.
720.0  Это водопад.
940.0  Открываю шаблон
640.0  Это дорога.

740.0  Это крёст.
1020.0  Продолжение следует...
980.0  Субтитры сделал DimaTorzok
1300.0  Субтитры сделал DimaTorzok
860.0  Вот и мы в шоке.
780.0  Это пикшо

In [ ]:
large_act_predictions2 = []

for audio in audios_actions:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.4,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_act_predictions2.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1060.0  Это мальчик? О, нет.
760.0  Точка пишет.
3460.0  Субтитры создавал DimaTorzok
960.0  пожарный тушит
1060.0  тут рабочие взглянут
860.0  Тут бабушка думает.

880.0  Так, я думаю, что греется.
820.0  тут мальчик надувает
860.0  Тут девочка нюхает.
980.0  Тут тетрадь плачет. Сейчас тоже тетрадь?
1060.0  Тут малыш играет. И сейчас, да?
680.0  тут мама пылесосит
760.0  Тут все серебро.
840.0  Тут мало кладет. Сейчас тоже.
780.0  тут папа стрижет
880.0  Тут корова мочит.
1200.0  Субтитры подогнал «Симон»
1040.0  Тут артист поет.
920.0  Тут бабушка кормит.
820.0  Тут пара консульт.
1000.0  Ча-ча-ча-ча...
840.0  девочка кушает
840.0  тут малыш падает
900.0  Тут бабушка в этот ряд.
880.0  Тут кость кричит.
840.0  Мальчик пьет.
680.0  Тут дочка пишет.
2640.0  Смотрите продолжение в следующей серии.
540.0  Что-то со мной нарезает.
780.0  Пусторус подует.
860.0  тут папа зажигает
840.0  Тут мальчик слушает.
1060.0  Тут сломано, лёд взлетает.
920.0  Тут рабочий кост.
880.0  Тут старик сеет.

In [ ]:
large_obj_predictions2 = list(map(clean, large_obj_predictions2))
print(large_obj_predictions2)

['это бан', 'это кроссворд', '', 'это верблюд', 'вот так вот', '', 'это дым и лом', 'это перчатка', 'это аквариум', 'это наручники', 'это пистолет', 'это кастыль', 'это страус', 'это корзина', '', 'это кактус', 'как ты видишь', '', 'это колеса', '', 'это примерно 2', 'это же рэп', 'это барабан', 'это башня', 'это вертолет', '', 'это фильм', 'это микрофон', 'поперек нет', 'вот тебе кружка', 'это пылесос', 'это ключ', 'это все', 'это както так', 'это мас', 'это суть', 'это кунгур', 'это водопад', 'открываю шаблон', 'это дорога', '', 'это крест', '', '', '', 'вот и мы в шоке', 'это пикшот', 'попробовать', 'это краска', 'это качаю']


In [ ]:
large_act_predictions2 = list(map(clean, large_act_predictions2))
print(large_act_predictions2)

['это мальчик о нет', 'точка пишет', '', 'пожарный тушит', 'тут рабочие взглянут', 'тут бабушка думает', '', 'так я думаю что греется', 'тут мальчик надувает', 'тут девочка нюхает', 'тут тетрадь плачет сейчас тоже тетрадь', 'тут малыш играет и сейчас да', 'тут мама пылесосит', 'тут все серебро', 'тут мало кладет сейчас тоже', 'тут папа стрижет', 'тут корова мочит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', '', 'девочка кушает', 'тут малыш падает', 'тут бабушка в этот ряд', 'тут кость кричит', 'мальчик пьет', 'тут дочка пишет', '', 'чтото со мной нарезает', 'пусторус подует', 'тут папа зажигает', 'тут мальчик слушает', 'тут сломано лед взлетает', 'тут рабочий кост', 'тут старик сеет', 'тут птица летает', 'тут и брат плавит', 'питер больше пилит', 'тут дочка режет', 'тут вся старая болеет', 'тут фигуру еще накрасить', 'судьба мальчику стучит', 'дочка чистит', 'пусть субтитры целуются', 'тут матрас стоит', 'тут меня не моют', 'просто ручка плевает', 'тут кивнушка ж

In [ ]:
obj_wer = wer_metric.compute(predictions=large_obj_predictions2, references=references_objects)
act_wer = wer_metric.compute(predictions=large_act_predictions2, references=references_actions)
obj_cer = cer_metric.compute(predictions=large_obj_predictions2, references=references_objects)
act_cer = cer_metric.compute(predictions=large_act_predictions2, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 62.4%
act_wer: 49.7%
obj_cer: 44.0%
act_cer: 32.1%


In [ ]:
large_obj_predictions3 = []

for audio in audios_objects:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.6,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_obj_predictions3.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

620.0  Это было.
620.0  Это кроссворд.
720.0  Редактор субтитров А.Семкин Корректор А.Егорова
540.0  Это верблюд.
520.0  Вот так вот.
1040.0  Продолжение следует...
780.0  Это вымыл он.
480.0  это перчатка
660.0  Это аквариум.
620.0  Это наручники.
480.0  Это пистолет.
700.0  Это кастыль.
680.0  Это страус.
620.0  Это корзина.
700.0  Спасибо за просмотр!
660.0  Это кактус.
640.0  как ты видишь
0.0  Спасибо.
540.0  это колеса
600.0  Спасибо за внимание!
660.0  Это примерно 2.
520.0  Это же рэп.
740.0  Это барабан.
660.0  Это башня.
540.0  Это вертолет.
600.0  Это шоу.
640.0  Это фильм.
560.0  Это микрофон.
800.0  Субтитры сделал DimaTorzok
600.0  Вот тебе кружка.
660.0  Это пылесос.
600.0  Это ключ.
540.0  Это что?
620.0  Это как-то так.
760.0  Это МАЗ.
660.0  Это суть.
620.0  Это кунгур.
720.0  Это водопад.

640.0  Это дорога.
900.0  Субтитры сделал DimaTorzok
740.0  Это крёст.
640.0  это кастрюля
980.0  Субтитры сделал DimaTorzok
1300.0  Субтитры сделал DimaTorzok
860.0  вот и мыс шоу

In [ ]:
large_act_predictions3 = []

for audio in audios_actions:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.6,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_act_predictions3.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1040.0  Мальчик улет.
760.0  Точка пишет.
1520.0  Субтитры сделал DimaTorzok
980.0  Пожарный тушат.
0.0  рабочие
860.0  Тут бабушка думает.

1480.0  Так, вернемся к границе.
820.0  тут мальчик надувает
860.0  Тут девочка нюхает.
980.0  Тут тетрадь плачет. Сейчас тоже трудно?
1060.0  Тут малыш играет. И сейчас, да?
680.0  тут мама пылесосит

840.0  Тут мало кладет. Сейчас тоже.
780.0  тут папа стрижет
880.0  Тут корова мычит.
1080.0  Редактор субтитров А.Семкин Корректор А.Егорова
1040.0  Тут артист поет.
920.0  тут бабушка кормит
820.0  Тут пара консульт.
940.0  Сейчас же она копает.
840.0  девочка кушает
840.0  тут малыш падает
900.0  Тут бабушка в этот ряд.
880.0  тут кость кричит
840.0  Мальчик льёт.
680.0  тут дочка пишет
2640.0  Смотрите продолжение в следующей серии.
540.0  Что-то с нами наезжает.
840.0  Тусторушка дует.
860.0  тут папа зажигает
840.0  Тут мальчик слушает.
660.0  Тут, в самом деле, лёд взлетает.
920.0  Тут рабочий кост.
880.0  Тут старик сеет.
900.0  тут птица ле

In [ ]:
large_obj_predictions3 = list(map(clean, large_obj_predictions3))
print(large_obj_predictions3)

['это было', 'это кроссворд', '', 'это верблюд', 'вот так вот', '', 'это вымыл он', 'это перчатка', 'это аквариум', 'это наручники', 'это пистолет', 'это кастыль', 'это страус', 'это корзина', '', 'это кактус', 'как ты видишь', 'спасибо', 'это колеса', '', 'это примерно 2', 'это же рэп', 'это барабан', 'это башня', 'это вертолет', 'это шоу', 'это фильм', 'это микрофон', '', 'вот тебе кружка', 'это пылесос', 'это ключ', 'это что', 'это както так', 'это маз', 'это суть', 'это кунгур', 'это водопад', '', 'это дорога', '', 'это крест', 'это кастрюля', '', '', 'вот и мыс шоу', 'это пиджак', 'попробовать', '', 'это качай']


In [ ]:
large_act_predictions3 = list(map(clean, large_act_predictions3))
print(large_act_predictions3)

['мальчик улет', 'точка пишет', '', 'пожарный тушат', 'рабочие', 'тут бабушка думает', '', 'так вернемся к границе', 'тут мальчик надувает', 'тут девочка нюхает', 'тут тетрадь плачет сейчас тоже трудно', 'тут малыш играет и сейчас да', 'тут мама пылесосит', '', 'тут мало кладет сейчас тоже', 'тут папа стрижет', 'тут корова мычит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', 'сейчас же она копает', 'девочка кушает', 'тут малыш падает', 'тут бабушка в этот ряд', 'тут кость кричит', 'мальчик льет', 'тут дочка пишет', '', 'чтото с нами наезжает', 'тусторушка дует', 'тут папа зажигает', 'тут мальчик слушает', 'тут в самом деле лед взлетает', 'тут рабочий кост', 'тут старик сеет', 'тут птица летает', 'тут и брат поет', 'питер больше пилит', 'тут дочка режет', 'тут все старо болеет', 'тут лучше красить', 'тут был мальчик стучит', 'тут дочка чистит', 'пусть субтитры соединятся', 'тут матрас стоит', 'тут меня не моют', '', 'тут кивнушка ждет', 'тут брат собирает']


In [ ]:
obj_wer = wer_metric.compute(predictions=large_obj_predictions3, references=references_objects)
act_wer = wer_metric.compute(predictions=large_act_predictions3, references=references_actions)
obj_cer = cer_metric.compute(predictions=large_obj_predictions3, references=references_objects)
act_cer = cer_metric.compute(predictions=large_act_predictions3, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 59.4%
act_wer: 50.3%
obj_cer: 44.5%
act_cer: 33.0%


Лучшие показатели наблюдаются при температуре, равной 0 или 0.2

Попробуем поизменять гиперпараметр best_of. Увеличим до 10:

In [ ]:
large_obj_predictions7 = []

for audio in audios_objects:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.2,
      best_of=10,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_obj_predictions7.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

620.0  Это бан.
620.0  это кроссворд
880.0  Продолжение следует...
540.0  Это верблюд.
520.0  Вот так вот.
0.0  пружину.
780.0  Это дым и лом.
480.0  это перчатка
660.0  Это аквариум.
620.0  Это наручники.
420.0  Тут и пистолет.
700.0  Это кастыль.
680.0  Это страус.
620.0  Это корзина.
620.0  Это фитбол.
660.0  Это кактус.
640.0  как ты видишь
0.0  Спасибо.
540.0  это колеса
3440.0  Субтитры сделал DimaTorzok
660.0  Это примерно 2.
520.0  Это же рэп.
740.0  Это барабан.
660.0  Это башня.
540.0  Это вертолёт.
600.0  Это шоу.
640.0  Это фильм.
560.0  Это микрофон.
920.0  Поперек нет.
600.0  Вот тебе кружка.
660.0  Это пылесос.
600.0  Это ключ.
540.0  Это все.
620.0  Это как-то так.
760.0  Это МАС.
660.0  Это суть.
620.0  Это кунгур.
720.0  Это водопад.
820.0  оттуда уже белка
640.0  Это дорога.
880.0  Продолжение следует...
740.0  Это крёст.
640.0  Это то стрельба.
960.0  Продолжение следует...
760.0  Вот и все.
860.0  Вот и мы в шоке.
780.0  Это пикшот.
0.0  попробовать
660.0  Это крас

In [ ]:
large_act_predictions7 = []

for audio in audios_actions:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.2,
      best_of=10,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_act_predictions7.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1060.0  Это мальчик? О, нет.
760.0  Точка пишет.
1520.0  Субтитры сделал DimaTorzok
980.0  Пожарный тушит.
1060.0  тут рабочие вслепят
860.0  Тут бабушка думает.

1480.0  только вернувшую границу
820.0  тут мальчик надувает
860.0  Тут девочка нюхает.
980.0  Тут тетрадь плачет. Сейчас тоже тетрадь?
1060.0  Тут малыш играет. И сейчас, да?
680.0  тут мама пылесосит
760.0  Тут все серебро.
840.0  Тут мало кладет. Сейчас тоже.
780.0  тут папа стрижет
880.0  Тут корова мочит.
1240.0  Субтитры сделал DimaTorzok
1040.0  Тут артист поет.
920.0  Тут бабушка кормит.
820.0  Тут пара консульт.
1040.0  Ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ч Ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-

In [ ]:
large_obj_predictions7 = list(map(clean, large_obj_predictions7))
print(large_obj_predictions7)

['это бан', 'это кроссворд', '', 'это верблюд', 'вот так вот', 'пружину', 'это дым и лом', 'это перчатка', 'это аквариум', 'это наручники', 'тут и пистолет', 'это кастыль', 'это страус', 'это корзина', 'это фитбол', 'это кактус', 'как ты видишь', 'спасибо', 'это колеса', '', 'это примерно 2', 'это же рэп', 'это барабан', 'это башня', 'это вертолет', 'это шоу', 'это фильм', 'это микрофон', 'поперек нет', 'вот тебе кружка', 'это пылесос', 'это ключ', 'это все', 'это както так', 'это мас', 'это суть', 'это кунгур', 'это водопад', 'оттуда уже белка', 'это дорога', '', 'это крест', 'это то стрельба', '', 'вот и все', 'вот и мы в шоке', 'это пикшот', 'попробовать', 'это краска', 'это качай']


In [ ]:
large_act_predictions7 = list(map(clean, large_act_predictions7))
print(large_act_predictions7)

['это мальчик о нет', 'точка пишет', '', 'пожарный тушит', 'тут рабочие вслепят', 'тут бабушка думает', '', 'только вернувшую границу', 'тут мальчик надувает', 'тут девочка нюхает', 'тут тетрадь плачет сейчас тоже тетрадь', 'тут малыш играет и сейчас да', 'тут мама пылесосит', 'тут все серебро', 'тут мало кладет сейчас тоже', 'тут папа стрижет', 'тут корова мочит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', 'ч', 'девочка кушает', 'тут малыш падает', 'тут бабушка в этот ряд', 'тут кость кричит', 'мальчик пьет', 'тут дочка пищит', '', 'чтото с ним наряжается', 'пусторус подует', 'тут папа зажигает', 'тут мальчик слушает', 'тут сломанная лездовица летает', 'тут рабочий кост', 'тут старик сеет', 'тут птица летает', 'тут и брат плавит', 'питер больше пилит', 'тут дочка режет', 'тут все старо болеет', 'тут хинуша красит', 'тут бы обманщик стучит', 'тут дочка чистит', 'пусть субтитры целуются', 'тут матрас стоит', 'тут меня не моют', 'и старушка плевает', 'тут кивнушка 

In [ ]:
obj_wer = wer_metric.compute(predictions=large_obj_predictions7, references=references_objects)
act_wer = wer_metric.compute(predictions=large_act_predictions7, references=references_actions)
obj_cer = cer_metric.compute(predictions=large_obj_predictions7, references=references_objects)
act_cer = cer_metric.compute(predictions=large_act_predictions7, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 64.4%
act_wer: 47.2%
obj_cer: 41.3%
act_cer: 30.5%


In [ ]:
large_obj_predictions4 = []

for audio in audios_objects:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.4,
      best_of=10,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_obj_predictions4.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

620.0  Это было.
620.0  это кроссворд
880.0  Продолжение следует...
540.0  Это верблюд.
520.0  Вот так вот.
500.0  Это пружина.
780.0  Это дым и лоб.
480.0  это перчатка
660.0  Это аквариум.
620.0  Это наручники.
420.0  Тут и пистолет.
700.0  Это кастер.
680.0  Это страус.
620.0  Это корзина.
800.0  Продолжение следует...
660.0  Это кактус.
640.0  как ты видишь
0.0  Спасибо.
540.0  это колеса
640.0  Редактор субтитров А.Семкин Корректор А.Егорова
660.0  Это примерно 2.
520.0  Это же рэп.
740.0  Это барабан.
660.0  Это башня.
540.0  Это вертолёт.
600.0  Это шоу.
640.0  Это фильм.
560.0  Это микрофон.
960.0  Копируйте это.
600.0  Вот тебе кружка.
660.0  Это пылесос.
600.0  Это ключ.
540.0  Это все.
620.0  Это как-то так.
760.0  Это МАС.
660.0  Это суть.
620.0  Это кунгур.
720.0  Это водопад.
820.0  оттуда уже белка
640.0  Это дорога.
880.0  Продолжение следует...
740.0  Это крёст.
640.0  Это кастрюля.
980.0  Субтитры сделал DimaTorzok
1300.0  Субтитры сделал DimaTorzok
860.0  Вот и мы в 

In [ ]:
large_act_predictions4 = []

for audio in audios_actions:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.4,
      best_of=10,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_act_predictions4.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1060.0  Это мальчик? О, нет.
760.0  Точка пишет.
1520.0  Субтитры сделал DimaTorzok
960.0  пожарный тушит
1060.0  тут рабочие вслепятся
860.0  Тут бабушка думает.
540.0  тут сестра
880.0  только не умею, что греется
820.0  тут мальчик надувает
860.0  Тут девочка нюхает.
980.0  Тут тетрадь плачет. Сейчас тоже тетрадь?
1060.0  Тут Малыш играет. И сейчас, да? Угу.
680.0  тут мама пылесосит
760.0  Тут все серебро.
840.0  Тут мало кладет. Сейчас тоже.
780.0  тут папа стрижет
880.0  Тут корова мочит.
1240.0  Субтитры сделал DimaTorzok
1040.0  Тут артист поет.
920.0  Тут бабушка кормит.
820.0  Тут пара консульт.
1040.0  Ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ч ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча-ча

In [ ]:
large_obj_predictions4 = list(map(clean, large_obj_predictions4))
print(large_obj_predictions4)

['это было', 'это кроссворд', '', 'это верблюд', 'вот так вот', 'это пружина', 'это дым и лоб', 'это перчатка', 'это аквариум', 'это наручники', 'тут и пистолет', 'это кастер', 'это страус', 'это корзина', '', 'это кактус', 'как ты видишь', 'спасибо', 'это колеса', '', 'это примерно 2', 'это же рэп', 'это барабан', 'это башня', 'это вертолет', 'это шоу', 'это фильм', 'это микрофон', 'копируйте это', 'вот тебе кружка', 'это пылесос', 'это ключ', 'это все', 'это както так', 'это мас', 'это суть', 'это кунгур', 'это водопад', 'оттуда уже белка', 'это дорога', '', 'это крест', 'это кастрюля', '', '', 'вот и мы в шоке', 'это пиджак', 'попробовать', 'это краска', 'это качай']


In [ ]:
large_act_predictions4 = list(map(clean, large_act_predictions4))
print(large_act_predictions4)

['это мальчик о нет', 'точка пишет', '', 'пожарный тушит', 'тут рабочие вслепятся', 'тут бабушка думает', 'тут сестра', 'только не умею что греется', 'тут мальчик надувает', 'тут девочка нюхает', 'тут тетрадь плачет сейчас тоже тетрадь', 'тут малыш играет и сейчас да угу', 'тут мама пылесосит', 'тут все серебро', 'тут мало кладет сейчас тоже', 'тут папа стрижет', 'тут корова мочит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', 'ч', 'девочка кушает', 'тут малыш падает', 'тут бабушка в этот ряд', 'тут кость кричит', 'мальчик пьет', 'тут дочка пищит', '', 'чтото со мной наряжается', 'пусть та рука дует', 'тут папа зажигает', 'тут мальчик слушает', 'тут самолет взлетает', 'тут рабочий кост', 'тут старик сеет', 'тут птица летает', 'тут и брат плавит', 'питер больше пилит', 'тут дочка режет', 'твоя сестра болеет', 'тут фигуру еще накрасить', 'куда мальчик стучит', 'тут дочка чистит', 'пусть субтитры целуются', 'тут матрас стоит', 'тут меня не моют', 'и старушка плевает',

In [ ]:
obj_wer = wer_metric.compute(predictions=large_obj_predictions4, references=references_objects)
act_wer = wer_metric.compute(predictions=large_act_predictions4, references=references_actions)
obj_cer = cer_metric.compute(predictions=large_obj_predictions4, references=references_objects)
act_cer = cer_metric.compute(predictions=large_act_predictions4, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 60.4%
act_wer: 47.8%
obj_cer: 40.6%
act_cer: 30.3%


Видим, что по всем метрикам значения улучшились, делая эту комбинацию пока что самой удачной. Тем не менее, попробуем ещё другие гиперпараметры.

Нулевую температуру с best_of=10 смысла пробовать не имеет, так как изменения параметра best_of проявляют себя только при ненулевой температуре.

Попробуем увеличить no_speech_threshold и добавить новый параметр condition_on_previous_text=False

In [ ]:
large_obj_predictions5 = []

for audio in audios_objects:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.8,
      condition_on_previous_text=False,
      word_timestamps=True
  )

  large_obj_predictions5.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

620.0  Это было.
620.0  это кроссворд
1060.0  Субтитры сделал DimaTorzok
540.0  Это верблюд.
520.0  Вот так вот.
500.0  Это пружина.
780.0  Это вымыл он.
480.0  это перчатка
660.0  Это аквариум.
620.0  Это наручники.
420.0  Тут и пистолет.
700.0  Это кастыль.
680.0  Это страус.
620.0  Это корзина.
840.0  Субтитры сделал DimaTorzok
660.0  Это кактус.
640.0  как ты видишь
3500.0  Субтитры сделал DimaTorzok
540.0  это колеса
3440.0  Субтитры сделал DimaTorzok
940.0  Продолжение следует...
520.0  Это же рэп.
740.0  Это барабан.
660.0  Это башня.
540.0  Это вертолёт.
600.0  Это шоу.
640.0  Это фильм.
560.0  Это микрофон.
920.0  Поперек нет.
600.0  Вот тебе кружка.
660.0  Это пылесос.
600.0  Это ключ.
540.0  Это все.
660.0  Вот такая ситуация.
760.0  Это МАС.
660.0  Это суть.
620.0  Это кунгур.
720.0  Это водопад.
820.0  оттуда уже белка
640.0  Это дорога.
900.0  Субтитры сделал DimaTorzok
740.0  Это крёст.
640.0  Это кастрюля.
960.0  Продолжение следует...
1300.0  Субтитры сделал DimaTorzok

In [ ]:
large_act_predictions5 = []

for audio in audios_actions:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.8,
      condition_on_previous_text=False,
      word_timestamps=True
  )

  large_act_predictions5.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1040.0  Мальчик улет.
900.0  И точка пишет.
1520.0  Субтитры сделал DimaTorzok
980.0  Пожарный тушит.
1060.0  тут рабочие вслепят
860.0  Тут бабушка думает.
540.0  тут сестра
880.0  Так, я думаю, что греется.
820.0  тут мальчик надувает
860.0  Тут девочка нюхает.
980.0  Тут тетрадь плачет. Сейчас тоже тетрадь?
1060.0  Тут малыш играет. И сейчас, да?
680.0  тут мама пылесосит
760.0  Тут все серебро.
840.0  Тут мало кладет. Сейчас тоже.
820.0  Тут папа стрижет.
880.0  Тут корова мочит.
1240.0  Субтитры сделал DimaTorzok
1040.0  Тут артист поет.
920.0  Тут бабушка кормит.
820.0  Тут пара консульт.
1000.0  Ча-ча-ча-ча-ча-ча
520.0  Какая-то девочка кушает.
840.0  тут малыш падает
900.0  Тут бабушка в этот ряд.
880.0  Тут кость кричит.
840.0  Мальчик пьет.
680.0  Тут дочка пищит.
2640.0  Смотрите продолжение в следующей серии.
540.0  Что-то со мной наряжается.
840.0  Пусторушка дует.
860.0  Тут папа зажигает.
840.0  Тут мальчик слушает.
1020.0  Тут сломанная лёздовица летает.
920.0  Тут рабо

In [ ]:
large_obj_predictions5 = list(map(clean, large_obj_predictions5))
print(large_obj_predictions5)

['это было', 'это кроссворд', '', 'это верблюд', 'вот так вот', 'это пружина', 'это вымыл он', 'это перчатка', 'это аквариум', 'это наручники', 'тут и пистолет', 'это кастыль', 'это страус', 'это корзина', '', 'это кактус', 'как ты видишь', '', 'это колеса', '', '', 'это же рэп', 'это барабан', 'это башня', 'это вертолет', 'это шоу', 'это фильм', 'это микрофон', 'поперек нет', 'вот тебе кружка', 'это пылесос', 'это ключ', 'это все', 'вот такая ситуация', 'это мас', 'это суть', 'это кунгур', 'это водопад', 'оттуда уже белка', 'это дорога', '', 'это крест', 'это кастрюля', '', '', 'вот и мы в шоке', 'это пикшок', 'попробовать', 'это краска', 'это качай']


In [ ]:
large_act_predictions5 = list(map(clean, large_act_predictions5))
print(large_act_predictions5)

['мальчик улет', 'и точка пишет', '', 'пожарный тушит', 'тут рабочие вслепят', 'тут бабушка думает', 'тут сестра', 'так я думаю что греется', 'тут мальчик надувает', 'тут девочка нюхает', 'тут тетрадь плачет сейчас тоже тетрадь', 'тут малыш играет и сейчас да', 'тут мама пылесосит', 'тут все серебро', 'тут мало кладет сейчас тоже', 'тут папа стрижет', 'тут корова мочит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', 'чачачачачача', 'какаято девочка кушает', 'тут малыш падает', 'тут бабушка в этот ряд', 'тут кость кричит', 'мальчик пьет', 'тут дочка пищит', 'смотрите продолжение в следующей серии', 'чтото со мной наряжается', 'пусторушка дует', 'тут папа зажигает', 'тут мальчик слушает', 'тут сломанная лездовица летает', 'тут рабочий кост', 'тут старик сеет', 'тут птица летает', 'тут и брат плавит', 'питер больше пилит', 'тут дочка режет', 'тут сестра болеет', 'тут фигуру еще накрасить', 'судьба мальчику стучит', 'тут дочка чистит', '', 'тут матрас стоит', 'тут меня 

In [ ]:
obj_wer = wer_metric.compute(predictions=large_obj_predictions5, references=references_objects)
act_wer = wer_metric.compute(predictions=large_act_predictions5, references=references_actions)
obj_cer = cer_metric.compute(predictions=large_obj_predictions5, references=references_objects)
act_cer = cer_metric.compute(predictions=large_act_predictions5, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 61.4%
act_wer: 47.2%
obj_cer: 43.2%
act_cer: 31.2%


Как видим, результат мало чем отличается от результата при дефолтных значениях параметров, поэтому лучше между ними двумя оставим дефолтный вариант

Наконец, попробуем увеличить значение гиперпараметра beam_size до 10 (температура для этого должна быть равна 0):

In [ ]:
large_obj_predictions6 = []

for audio in audios_objects:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=10,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_obj_predictions6.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

620.0  Это было.
620.0  это кроссворд
880.0  Продолжение следует...
540.0  Это верблюд.
520.0  Вот так вот.
500.0  Это пружина.
780.0  Это вымыл он.
480.0  это перчатка
660.0  Это аквариум.
620.0  Это наручники.
420.0  Тут и пистолет.
1180.0  Продолжение следует...
680.0  Это страус.
620.0  Это корзина.
840.0  Субтитры сделал DimaTorzok
660.0  Это кактус.
640.0  как ты видишь
3500.0  Субтитры сделал DimaTorzok
540.0  это колеса
3440.0  Субтитры сделал DimaTorzok
740.0  Субтитры сделал DimaTorzok
520.0  Это же рэп.
740.0  Это барабан.
660.0  Это башня.
540.0  Это вертолёт.
600.0  Это шоу.
640.0  Это фильм.
560.0  Это микрофон.
800.0  Субтитры сделал DimaTorzok
600.0  Вот тебе кружка.
660.0  Это пылесос.
600.0  Это ключ.
540.0  Это все.
800.0  Субтитры сделал DimaTorzok
760.0  Это МАС.
1120.0  Субтитры сделал DimaTorzok
620.0  Это кунгур.
720.0  Это водопад.
820.0  оттуда уже белка
640.0  Это дорога.
900.0  Субтитры сделал DimaTorzok
740.0  Это крёст.
1020.0  Продолжение следует...
980.0

In [ ]:
large_act_predictions6 = []

for audio in audios_actions:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=10,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  large_act_predictions6.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

1060.0  Это мальчик? О, нет.
900.0  И точка пишет.
1520.0  Субтитры сделал DimaTorzok
980.0  Пожарный тушит.
580.0  Какой-то рабочий вслепит.
860.0  Тут бабушка думает.
540.0  тут сестра
1400.0  Я думаю, что греется.
820.0  тут мальчик надувает
860.0  Тут девочка нюхает.
980.0  Тут тетрадь плачет. Сейчас тоже тетрадь?
1060.0  Тут малыш играет. И сейчас, да?
680.0  тут мама пылесосит
1080.0  Субтитры сделал DimaTorzok
840.0  Тут мало кладет. Сейчас тоже.
820.0  Тут папа стрижет.
880.0  Тут корова мочит.
1240.0  Субтитры сделал DimaTorzok
1040.0  Тут артист поет.
920.0  Тут бабушка кормит.
820.0  Тут пара консульт.
1000.0  Ча-ча-ча-ча-ча-ча
520.0  Какая-то девочка кушает.
840.0  тут малыш падает
900.0  Тут бабушка в этот ряд.
880.0  Тут кость кричит.
840.0  Мальчик пьет.
680.0  Тут дочка пищит.
2640.0  Смотрите продолжение в следующей серии.
540.0  Что-то со мной наряжается.
840.0  Тусторушка дует.
860.0  Тут папа зажигает.
840.0  Тут мальчик слушает.
1020.0  Тут самолет взлетает.
920.0 

In [ ]:
large_obj_predictions6 = list(map(clean, large_obj_predictions6))
print(large_obj_predictions6)

['это было', 'это кроссворд', '', 'это верблюд', 'вот так вот', 'это пружина', 'это вымыл он', 'это перчатка', 'это аквариум', 'это наручники', 'тут и пистолет', '', 'это страус', 'это корзина', '', 'это кактус', 'как ты видишь', '', 'это колеса', '', '', 'это же рэп', 'это барабан', 'это башня', 'это вертолет', 'это шоу', 'это фильм', 'это микрофон', '', 'вот тебе кружка', 'это пылесос', 'это ключ', 'это все', '', 'это мас', '', 'это кунгур', 'это водопад', 'оттуда уже белка', 'это дорога', '', 'это крест', '', '', '', 'вот и мы в шоке', 'это пикшок', 'попробовать', 'это краска', 'это качай']


In [ ]:
large_act_predictions6 = list(map(clean, large_act_predictions6))
print(large_act_predictions6)

['это мальчик о нет', 'и точка пишет', '', 'пожарный тушит', 'какойто рабочий вслепит', 'тут бабушка думает', 'тут сестра', 'я думаю что греется', 'тут мальчик надувает', 'тут девочка нюхает', 'тут тетрадь плачет сейчас тоже тетрадь', 'тут малыш играет и сейчас да', 'тут мама пылесосит', '', 'тут мало кладет сейчас тоже', 'тут папа стрижет', 'тут корова мочит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', '', 'какаято девочка кушает', 'тут малыш падает', 'тут бабушка в этот ряд', 'тут кость кричит', 'мальчик пьет', 'тут дочка пищит', '', 'чтото со мной наряжается', 'тусторушка дует', 'тут папа зажигает', 'тут мальчик слушает', 'тут самолет взлетает', 'тут рабочий кост', 'тут старик сеет', 'тут птица летает', 'тут и брат плавит', 'питер больше пилит', 'тут дочка режет', 'твоя сестра болеет', 'тут фигуру еще накрасить', 'судьба мальчику стучит', 'тут дочка чистит', '', 'тут матрас стоит', 'тут меня не моют', 'просто ручка плевает', 'тут кивнушка ждет', 'тут брат соби

In [ ]:
obj_wer = wer_metric.compute(predictions=large_obj_predictions6, references=references_objects)
act_wer = wer_metric.compute(predictions=large_act_predictions6, references=references_actions)
obj_cer = cer_metric.compute(predictions=large_obj_predictions6, references=references_objects)
act_cer = cer_metric.compute(predictions=large_act_predictions6, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 63.4%
act_wer: 47.8%
obj_cer: 46.8%
act_cer: 31.7%


Итого, в результате изменения гиперпараметров вышло, что наименьшую ошибку по всем метрикам показывает модель со следующей комбинацией гиперпараметров: temperature=0.4, best_of=10. Значит, будем считать эту модель лучшей до дообучения

Попробуем применить к аудиозаписям базовое шумоподавление:

In [ ]:
from pydub import AudioSegment

def clean_audio(audio_path, output_path="boosted.wav"):
    audio, sampling_rate = librosa.load(audio_path, sr=16000, mono=True)
    audio_denoised = librosa.effects.preemphasis(audio)
    sf.write(output_path, audio_denoised, sampling_rate)

    result = model_large.transcribe(
      output_path,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.4,
      best_of=10,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True,
      verbose=False
    )


    return result

In [ ]:
large_obj_predictions8 = []

for audio in audios_objects:
  result = clean_audio(audio, output_path="cleaned.wav")

  large_obj_predictions8.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

100%|██████████| 400/400 [00:01<00:00, 247.43frames/s]


620.0  Это банк.


100%|██████████| 400/400 [00:01<00:00, 202.35frames/s]


620.0  Это кроссворд.


100%|██████████| 400/400 [00:02<00:00, 199.74frames/s]


...

0.0  Попробуйте.


100%|██████████| 400/400 [00:01<00:00, 201.10frames/s]


1120.0  Продолжение следует...


100%|██████████| 400/400 [00:01<00:00, 238.28frames/s]

660.0  Это качает.


In [ ]:
large_act_predictions8 = []

for audio in audios_actions:
  result = clean_audio(audio, output_path="cleaned.wav")

  large_act_predictions8.append(result["text"])

  if result["segments"] and len(result["segments"][0]["words"]) >= 3:
    print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 2:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

100%|██████████| 400/400 [00:01<00:00, 254.38frames/s]


960.0  Значит, будет.


100%|██████████| 400/400 [00:01<00:00, 210.94frames/s]


780.0  Точка пишет.


100%|██████████| 400/400 [00:01<00:00, 210.94frames/s]


...

1180.0  Редактор субтитров А.Семкин Корректор А.Егорова


100%|██████████| 400/400 [00:02<00:00, 183.01frames/s]


920.0  Тут кивнушка ждёт.


100%|██████████| 400/400 [00:01<00:00, 239.25frames/s]

740.0  Тут брат собирает.


In [ ]:
large_obj_predictions8 = list(map(clean, large_obj_predictions8))
print(large_obj_predictions8)

['это банк', 'это кроссворд', '', 'это верблюд', '', 'пружина', 'это дымовое', 'это перчатка', 'это аквариум', 'это наручники', 'это пистолет', 'это костыль', 'это скрозь', 'это корзина', '', 'это кактус', 'это вершинка', '', 'это коляска', '', '', 'я тоже запрашиваю', 'это барабан', 'это башня', 'это вертолет', '', 'это все', 'это микрофон', '', '', 'это пылесос', 'это ключ', 'это шлем', 'это качество', 'это мас', '', 'это кунгфру', 'это вот такой вот', '', 'это дорога', '', 'это крест', '', '', '', 'вот он и шел', '', 'попробуйте', '', 'это качает']


In [ ]:
large_act_predictions8 = list(map(clean, large_act_predictions8))
print(large_act_predictions8)

['значит будет', 'точка пишет', '', 'пожар не тушит', 'тут рабочие вслепят', 'тут бабушка думает', '', 'так я думаю что греется', 'тут мальчик надувает', 'тут девочка мухает', 'тут тетрадь плачет сейчас тоже труп', 'тут малыш играет и сейчас да угу', 'тут мама пылесосит', 'тут ктото режет', 'тут мало кладет сейчас тоже', 'тут помоему стрижется', 'тут корова мочит', '', 'тут артист поет', 'тут бабушка кормит', 'тут пара консульт', 'ч', 'и девочка кушает', 'тут малыш падает', 'тут бабушка в этот ряд', 'тут кость кричит', 'мальчик льет', 'тут дочка пишет', '', 'чтото со мной нарезает', 'пусть старушка дует', 'тут папа зажигает', 'тут мальчик слушает', 'тут сломано лед взлетает', 'тут рабочий пост', 'тут старик сеет', 'тут птица летает', 'тут и брат ходит', 'тут рабочий пилит', 'тут дочка режет', 'ух ты сестра болеет', 'тут все лучше накрасить', 'пусть будет мальчик стучать', 'тут дочка чистит', 'пусть собрается солнце', 'тут матрас стоит', '', '', 'тут кивнушка ждет', 'тут брат собирает']

In [ ]:
obj_wer = wer_metric.compute(predictions=large_obj_predictions8, references=references_objects)
act_wer = wer_metric.compute(predictions=large_act_predictions8, references=references_actions)
obj_cer = cer_metric.compute(predictions=large_obj_predictions8, references=references_objects)
act_cer = cer_metric.compute(predictions=large_act_predictions8, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 61.4%
act_wer: 51.6%
obj_cer: 50.0%
act_cer: 35.0%


Попробуем Speech Recognition

In [ ]:
!pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 32.2 MB/s eta 0:00:00


In [ ]:
import speech_recognition as sr

r = sr.Recognizer()

Это при помощи Google Speech Recognition

In [ ]:
sr_objects = []

for audio_path in audios_objects:
  with sr.AudioFile(audio_path) as source:
    audio = r.record(source)

  try:
    text = r.recognize_google(audio, language="ru-RU")
    sr_objects.append(text)
    print(text)
  except sr.UnknownValueError:
    sr_objects.append('')
    print('')

это банк
это кроссворд
coca-cola
это верблюд

пружина
это домино
это перчатка
это аквариум
это наручники
это пистолет

это страус
это корзина
тюльпан
это кактус


это коляска

это помидор

это барабан
это башня

Photoshop
Это фон
Это микрофон
который пьёт
лягушка
это пылесос
это ключ
это шлем

Это Макс

Кто такой кенгуру
это водопад



Это кресло

бутерброд

фото мешок


это краска



In [ ]:
sr_actions = []

for audio_path in audios_actions:
  with sr.AudioFile(audio_path) as source:
    audio = r.record(source)

  try:
    text = r.recognize_google(audio, language="ru-RU")
    sr_actions.append(text)
    print(text)
  except sr.UnknownValueError:
    sr_actions.append('')
    print('')

этот Мальчик ходит


пожарный тушит
тут рабочие сверлят
Тут бабушка думает


тут Мальчик надувает
тут девочка нюхает
ретроград плачет
тут Малыш играет
тут мама пылесосит
тут тётя режет
тут мало слазит сейчас
тут папа стрижёт


тут артист поёт
Тут бабушка кормит
Тут парень танцует
капает
тут девочка кушает
малыш падает
Тут бабушка отряд
тупой стучит
матч Привет
цветочка пишет
окислитель

ты старушка дует
тут папа зажигает
тут мальчик слушает

тут рабочий косит
тут старик сеет
птица летает

тут рабочий пилит
тут дочка режет
болеет

мальчику стучит
дочка чистит


тут меня не моет
старушка поливает




In [ ]:
sr_objects = list(map(clean, sr_objects))
print(sr_objects)

['это банк', 'это кроссворд', 'cocacola', 'это верблюд', '', 'пружина', 'это домино', 'это перчатка', 'это аквариум', 'это наручники', 'это пистолет', '', 'это страус', 'это корзина', 'тюльпан', 'это кактус', '', '', 'это коляска', '', 'это помидор', '', 'это барабан', 'это башня', '', 'photoshop', 'это фон', 'это микрофон', 'который пьет', 'лягушка', 'это пылесос', 'это ключ', 'это шлем', '', 'это макс', '', 'кто такой кенгуру', 'это водопад', '', '', '', 'это кресло', '', 'бутерброд', '', 'фото мешок', '', '', 'это краска', '']


In [ ]:
sr_actions = list(map(clean, sr_actions))
print(sr_actions)

['этот мальчик ходит', '', '', 'пожарный тушит', 'тут рабочие сверлят', 'тут бабушка думает', '', '', 'тут мальчик надувает', 'тут девочка нюхает', 'ретроград плачет', 'тут малыш играет', 'тут мама пылесосит', 'тут тетя режет', 'тут мало слазит сейчас', 'тут папа стрижет', '', '', 'тут артист поет', 'тут бабушка кормит', 'тут парень танцует', 'капает', 'тут девочка кушает', 'малыш падает', 'тут бабушка отряд', 'тупой стучит', 'матч привет', 'цветочка пишет', 'окислитель', '', 'ты старушка дует', 'тут папа зажигает', 'тут мальчик слушает', '', 'тут рабочий косит', 'тут старик сеет', 'птица летает', '', 'тут рабочий пилит', 'тут дочка режет', 'болеет', '', 'мальчику стучит', 'дочка чистит', '', '', 'тут меня не моет', 'старушка поливает', '', '']


In [ ]:
obj_wer = wer_metric.compute(predictions=sr_objects, references=references_objects)
act_wer = wer_metric.compute(predictions=sr_actions, references=references_actions)
obj_cer = cer_metric.compute(predictions=sr_objects, references=references_objects)
act_cer = cer_metric.compute(predictions=sr_actions, references=references_actions)

print(f"obj_wer: {obj_wer * 100:.1f}%")
print(f"act_wer: {act_wer * 100:.1f}%")
print(f"obj_cer: {obj_cer * 100:.1f}%")
print(f"act_cer: {act_cer * 100:.1f}%")

obj_wer: 52.5%
act_wer: 55.3%
obj_cer: 45.8%
act_cer: 44.9%


Результаты у Whisper в целом лучше

Vosk здесь нет, потому что, как сказано в тексте работы, мне не удалось добиться от него даже нормального вывода транскрипций (скорее всего, связано с проблемами в распознавании записей)